# 4. Exploratory Data Analysis with SQL

**Goal:** answer ten questions about the launch records with SQL queries. The data is loaded into an in-memory SQLite database, and each query runs through pandas so the result prints as a table.

In [1]:
import re
import sqlite3

import pandas as pd

raw = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
# Some versions of this file have stray spaces in column names, for example "Landing _Outcome"
raw.columns = [re.sub(r"\s*_\s*", "_", c.strip()) for c in raw.columns]

con = sqlite3.connect(":memory:")
raw.to_sql("SPACEXTBL", con, if_exists="replace", index=False)
con.execute("DROP TABLE IF EXISTS SPACEXTABLE")
con.execute("CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL")


def sql(query):
    return pd.read_sql(query, con)


sql("SELECT COUNT(*) AS launches FROM SPACEXTABLE")

,launches
0,101


In [2]:
sql("SELECT * FROM SPACEXTABLE LIMIT 3")

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt


## Task 1: Unique launch sites

In [3]:
sql("""
SELECT DISTINCT "Launch_Site" FROM SPACEXTABLE
""")

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


## Task 2: Five launches from sites that begin with 'CCA'

In [4]:
sql("""
SELECT * FROM SPACEXTABLE WHERE "Launch_Site" LIKE 'CCA%' LIMIT 5
""")

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


## Task 3: Total payload mass carried for NASA (CRS)

In [5]:
sql("""
SELECT SUM("PAYLOAD_MASS__KG_") AS total_payload_kg FROM SPACEXTABLE WHERE "Customer" = 'NASA (CRS)'
""")

,total_payload_kg
0,45596


## Task 4: Average payload mass carried by booster version F9 v1.1

In [6]:
sql("""
SELECT AVG("PAYLOAD_MASS__KG_") AS avg_payload_kg FROM SPACEXTABLE WHERE "Booster_Version" = 'F9 v1.1'
""")

,avg_payload_kg
0,2928.4


## Task 5: Date of the first successful landing on a ground pad

In [7]:
sql("""
SELECT MIN("Date") AS first_ground_pad_landing FROM SPACEXTABLE WHERE "Landing_Outcome" = 'Success (ground pad)'
""")

,first_ground_pad_landing
0,2015-12-22


## Task 6: Boosters that landed on a drone ship with a payload between 4,000 and 6,000 kg

In [8]:
sql("""
SELECT "Booster_Version" FROM SPACEXTABLE
WHERE "Landing_Outcome" = 'Success (drone ship)'
  AND "PAYLOAD_MASS__KG_" > 4000 AND "PAYLOAD_MASS__KG_" < 6000
""")

,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


## Task 7: Number of successful and failed mission outcomes

In [9]:
sql("""
SELECT "Mission_Outcome", COUNT(*) AS launches FROM SPACEXTABLE GROUP BY "Mission_Outcome"
""")

,Mission_Outcome,launches
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1


## Task 8: Booster versions that carried the maximum payload (subquery)

In [10]:
sql("""
SELECT DISTINCT "Booster_Version" FROM SPACEXTABLE
WHERE "PAYLOAD_MASS__KG_" = (SELECT MAX("PAYLOAD_MASS__KG_") FROM SPACEXTABLE)
""")

,Booster_Version
0,F9 B5 B1048.4
1,F9 B5 B1049.4
2,F9 B5 B1051.3
3,F9 B5 B1056.4
4,F9 B5 B1048.5
5,F9 B5 B1051.4
6,F9 B5 B1049.5
7,F9 B5 B1060.2
8,F9 B5 B1058.3
9,F9 B5 B1051.6


## Task 9: Failed drone-ship landings in 2015, by month

In [11]:
sql("""
SELECT substr("Date", 6, 2) AS month, "Landing_Outcome", "Booster_Version", "Launch_Site"
FROM SPACEXTABLE
WHERE substr("Date", 0, 5) = '2015' AND "Landing_Outcome" = 'Failure (drone ship)'
""")

,month,Landing_Outcome,Booster_Version,Launch_Site
0,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


## Task 10: Landing outcomes between 2010-06-04 and 2017-03-20, ranked by count

In [12]:
sql("""
SELECT "Landing_Outcome", COUNT(*) AS launches FROM SPACEXTABLE
WHERE "Date" BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY "Landing_Outcome"
ORDER BY launches DESC
""")

,Landing_Outcome,launches
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1
